# Treinamento dos Modelos


Este notebook realiza o treinamento de modelos de risco de crédito e registra os modelos treinados no Unity Catalog para gerenciamento centralizado.

In [0]:
%pip install databricks-sdk==0.36.0 mlflow==2.19.0 databricks-feature-store==0.17.0 scikit-learn==1.5.2
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
import plotly.express as px
from datetime import datetime

# ============================================================
# Preencha apenas estas variáveis (mesmo padrão do 01-Criacao-dados)
# ============================================================
CATALOGO = ""
SCHEMA = ""
PREFIXO = ""

identificador_schema = f"{CATALOGO}.{SCHEMA}"

# Criando tabela de treino

Nesta seção construímos o dataset que será utilizado no treinamento dos modelos:

| Etapa | Descrição |
| --- | --- |
| Leitura de features | Carregamos a tabela de features da **Feature Store**, garantindo rastreabilidade e reutilização das variáveis preditivas. |
| Criação do label | A coluna `defaulted` é derivada da tabela `credit_bureau_gold`: atribuímos **1** quando o atraso (`CREDIT_DAY_OVERDUE`) ultrapassa 60 dias e **0** caso contrário. |
| Join features + label | Unimos features e label pela chave `cust_id`, formando o dataset completo de treino. |

> **Nota:** O dataset resultante é bastante desbalanceado — a grande maioria dos clientes não está inadimplente.

In [0]:
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()
features_set = fs.read_table(name=f"{identificador_schema}.{PREFIXO}credit_decisioning_features")
display(features_set)

In [0]:
# Criando a coluna de label "defaulted": 1 se atraso > 60 dias, senão 0
credit_bureau_label = (
    spark.table(f"{identificador_schema}.{PREFIXO}credit_bureau_gold")
         .withColumn("defaulted", F.when(col("CREDIT_DAY_OVERDUE") > 60, 1)
                                        .otherwise(0))
         .select("cust_id", "defaulted")
)

# Como podemos ver, temos um dataset bastante desbalanceado
df = credit_bureau_label.groupBy('defaulted').count().toPandas()
px.pie(df, values='count', names='defaulted', title='Proporção de inadimplência')

In [0]:
training_dataset = credit_bureau_label.join(features_set, "cust_id", "inner")

## Balanceamento do dataset

Datasets desbalanceados fazem com que o modelo "aprenda" apenas a prever a classe majoritária. Para mitigar isso, aplicamos duas técnicas combinadas:

| Técnica | O que faz |
| --- | --- |
| **Oversampling** (classe minoritária) | Duplicamos os registros de clientes inadimplentes (`defaulted = 1`) para aumentar sua representatividade. |
| **Undersampling** (classe majoritária) | Reduzimos aleatoriamente os registros de clientes adimplentes (`defaulted = 0`) de modo que a proporção fique mais equilibrada. |

O resultado é salvo como tabela Delta (`credit_risk_train_df`) para ser reutilizado nas próximas etapas sem reprocessamento.

In [0]:
major_df = training_dataset.filter(col("defaulted") == 0)
minor_df = training_dataset.filter(col("defaulted") == 1)

# Duplicar as linhas da classe minoritária
oversampled_df = minor_df.union(minor_df)

# Reduzir a classe majoritária
undersampled_df = major_df.sample(oversampled_df.count() / major_df.count() * 3, 42)

# Combinar ambas
train_df = undersampled_df.unionAll(oversampled_df).drop('cust_id').na.fill(0)

# Salvar como tabela para uso pelo AutoML
tabela_treino = f"{identificador_schema}.{PREFIXO}credit_risk_train_df"
train_df.write.mode('overwrite').saveAsTable(tabela_treino)
train_df = spark.table(tabela_treino)

# Visualizar a proporção de inadimplência após balanceamento
px.pie(train_df.groupBy('defaulted').count().toPandas(), values='count', names='defaulted', title='Proporção de inadimplência (após balanceamento)')

   
# Treinamento

Aqui treinamos **5 modelos de classificação simultaneamente** em um loop, registrando cada experimento no **MLflow**:

| Modelo | Tipo | Característica principal |
| --- | --- | --- |
| Gradient Boosting Classifier | Ensemble (boosting) | Combina árvores sequenciais corrigindo erros anteriores |
| Random Forest Classifier | Ensemble (bagging) | Média de muitas árvores independentes — robusto a overfitting |
| Extra Trees Classifier | Ensemble (bagging) | Similar ao Random Forest, mas com splits aleatórios — mais rápido |
| AdaBoost Classifier | Ensemble (boosting) | Pesos adaptativos focam nos exemplos difíceis |
| Logistic Regression | Regressão logística regularizada | Combina penalidades L1 e L2 — bom baseline linear |

**Fluxo por modelo:**
1. Treina no split de treino (80%)
2. Avalia no split de teste (20%) com métricas Accuracy, ROC AUC e F1
3. Registra parâmetros, métricas e artefato do modelo no MLflow

Ao final, o melhor modelo (maior ROC AUC) é selecionado e registrado no **Unity Catalog** com o alias `prod`, ficando disponível para inferência em produção.

In [0]:
from databricks import feature_store
import mlflow
from mlflow import MlflowClient
from pyspark.sql.functions import col
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

mlflow.set_registry_uri('databricks-uc')
mlflow.set_experiment(f"/{PREFIXO}_treinamento")

In [0]:
# Carregar dados da tabela credit_risk_train_df
credit_risk_pdf = spark.table(f"{identificador_schema}.{PREFIXO}credit_risk_train_df").toPandas()

# Separar features e target (defaulted)
X_cr = credit_risk_pdf.drop(columns=["defaulted"])
y_cr = credit_risk_pdf["defaulted"]

# Split treino/teste
X_train_cr, X_test_cr, y_train_cr, y_test_cr = train_test_split(X_cr, y_cr, test_size=0.2, random_state=42)
print(f"Treino: {X_train_cr.shape[0]} | Teste: {X_test_cr.shape[0]}")
print(f"Target 'defaulted' - proporção positivos: {y_cr.mean():.2%}\n")

# 5 modelos de classificação
models_cr = {
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    "Random Forest":     RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "Extra Trees":       ExtraTreesClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "AdaBoost":          AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    "Logistic Regression": LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=1.0, max_iter=1000, random_state=42)
}

results_cr = []

for name, m in models_cr.items():
    with mlflow.start_run(run_name=f"{name}_credit_risk") as run:
        m.fit(X_train_cr, y_train_cr)
        y_pred_cr = m.predict(X_test_cr)
        y_prob_cr = m.predict_proba(X_test_cr)[:, 1]

        acc_cr = accuracy_score(y_test_cr, y_pred_cr)
        roc_cr = roc_auc_score(y_test_cr, y_prob_cr)
        f1_cr  = f1_score(y_test_cr, y_pred_cr)

        mlflow.log_params(m.get_params())
        mlflow.log_metric("accuracy",  acc_cr)
        mlflow.log_metric("roc_auc",   roc_cr)
        mlflow.log_metric("f1_score",  f1_cr)
        mlflow.sklearn.log_model(m, "model", input_example=X_test_cr.iloc[:5])

        results_cr.append({"Model": name, "Accuracy": round(acc_cr, 4), "ROC AUC": round(roc_cr, 4), "F1": round(f1_cr, 4), "run_id": run.info.run_id})
        print(f"{name}: Accuracy={acc_cr:.4f}, ROC AUC={roc_cr:.4f}, F1={f1_cr:.4f}")

print("\n--- Comparação (credit_risk_train_df) ---")
comparison_cr_df = pd.DataFrame(results_cr).sort_values("ROC AUC", ascending=False)
display(comparison_cr_df)

In [0]:
# Selecionar o melhor modelo com base no maior ROC AUC
best_model_row = comparison_cr_df.sort_values("ROC AUC", ascending=False).iloc[0]
best_model_name = best_model_row["Model"]
print(f"Melhor modelo: {best_model_name} (ROC AUC={best_model_row['ROC AUC']})")

In [0]:
# Registrar o modelo no Unity Catalog
model_name = f"{PREFIXO}_credit_classification"
catalog = CATALOGO
db = SCHEMA

# Recuperar o run_id do melhor modelo já logado — sem novo treino
best_run_id = comparison_cr_df.loc[comparison_cr_df["Model"] == best_model_name, "run_id"].values[0]

client = MlflowClient()
registered_model = mlflow.register_model(f'runs:/{best_run_id}/model', f"{catalog}.{db}.{model_name}")
client.set_registered_model_alias(name=f"{catalog}.{db}.{model_name}", alias="prod", version=registered_model.version)

print(f"Modelo registrado: {catalog}.{db}.{model_name} v{registered_model.version} (alias: prod)")

In [0]:
# Carregar modelo registrado no Unity Catalog
loaded_model = mlflow.sklearn.load_model(f"models:/{catalog}.{db}.{model_name}@prod")

# Batch inference sobre os dados de teste
predictions   = loaded_model.predict(X_test_cr)
probabilities = loaded_model.predict_proba(X_test_cr)[:, 1]

# Criar DataFrame com resultados
results_batch = pd.DataFrame({
    'actual':              y_test_cr.values,
    'predicted':           predictions,
    'probability_default': probabilities
})

print(f"Batch inference: {len(results_batch)} registros processados")
print(f"\nResumo das previsões:")
display(spark.createDataFrame(results_batch).describe())
display(spark.createDataFrame(results_batch.head(20)))